# Занятие 5. Методы квантовой механики

In [ ]:
# 📥 Установка пакетов и библиотек
! pip install numpy
! pip install plotly
! pip install nbformat>=4.2.0 

In [ ]:
# 🛠️ Загрузка необходимых пакетов в среду выполнения
import os
import sys
import numpy as np
import math
import plotly.graph_objects as go

# show() открывает просмотрщик ORCA прямо под ячейкой
REPO = "/home/jupyter/project/computational-chemistry-course"
sys.path.append(os.path.join(REPO, "tools"))
from orca_tools import show

In [ ]:
# 🐋 Инициализация ORCA
ORCA = "/opt/orca/orca"  # ORCA стабильно работает только если задать абсолютный путь
print("ORCA найдена:", os.path.isfile(ORCA))

# Библиотеки и переменные для ORCA
os.environ["PATH"] = "/opt/orca:/opt/openmpi/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = "/opt/orca:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OMPI_MCA_plm"] = "isolated"

# Рабочая папка для расчётов
os.chdir(os.path.join(REPO, "Занятия", "Занятие_5"))

# 📌 Глава 1. Одноточечный расчёт 

## 1.1 Полная энергия системы

Полная энергия (Total Energy), которую рассчитывает ORCA, характеризует энергетическое состояние заданной молекулярной системы в рамках выбранного метода квантовой химии, базисного набора и набора поправок. Абсолютные значения полной энергии сами по себе обычно малоинформативны. Чтобы оценить относительную устойчивость структур или энергетический эффект реакции, необходимо сравнивать энергии согласованным способом.

Основные правила сравнения:

* **Единый метод расчёта**. Все рассматриваемые системы следует рассчитывать одним методом, например PBE0 или B3LYP. Значения полной энергии, полученные разными методами, **нельзя** напрямую сравнивать между собой.

* **Единый базисный набор**. Для всех систем необходимо использовать один и тот же базисный набор. Различия в базисах могут приводить к систематическим ошибкам.

* **Материальный баланс**. Сравниваемые системы должны содержать одинаковое количество атомов каждого химического элемента и иметь одинаковый суммарный заряд, если речь идёт о сравнении изомеров или конформеров. При расчёте химических реакций состав реагентов и продуктов должен соответствовать стехиометрии реакции. Энергии систем разного состава нельзя напрямую сравнивать без учёта энергий всех компонентов, участвующих в процессе.

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "water_HF_def2-SVP")
os.makedirs(work_path, exist_ok=True) # Создать папку water_HF_def2-SVP
os.chdir(work_path)                   # Перейти в директорию main_path/water_HF_def2-SVP

input_file = """\
! HF def2-SVP PrintMOs PrintBasis
%pal nprocs 4 end

* xyz 0 1
O   0.000000   0.000000   0.117300
H   0.000000   0.757200  -0.469200
H   0.000000  -0.757200  -0.469200
*
"""

with open("water_HF_def2-SVP.inp", "w") as f: # Записать input_file в файл water_HF_def2-SVP.inp
    f.write(input_file)

!/opt/orca/orca water_HF_def2-SVP.inp > water_HF_def2-SVP.out --oversubscribe # Запустить расчёт ORCA

In [ ]:
show("water_HF_def2-SVP.out")

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "water_B3LYP_def2-SVP")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP PrintMOs PrintBasis
%pal nprocs 4 end

* xyz 0 1
O   0.000000   0.000000   0.117300
H   0.000000   0.757200  -0.469200
H   0.000000  -0.757200  -0.469200
*
"""

with open("water_B3LYP_def2-SVP.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca water_B3LYP_def2-SVP.inp > water_B3LYP_def2-SVP.out --oversubscribe

In [ ]:
show("water_B3LYP_def2-SVP.out")

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "water_B3LYP_def2-TZVP")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP PrintMOs PrintBasis
%pal nprocs 4 end

* xyz 0 1
O   0.000000   0.000000   0.117300
H   0.000000   0.757200  -0.469200
H   0.000000  -0.757200  -0.469200
*
"""

with open("water_B3LYP_def2-TZVP.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca water_B3LYP_def2-TZVP.inp > water_B3LYP_def2-TZVP.out --oversubscribe

In [ ]:
show("water_B3LYP_def2-TZVP.out")

## 1.2 Теорема Купманса

В методе Хартри-Фока энергия занятой орбитали приближённо связана с энергией отрыва электрона с неё. Согласно теореме Купманса, потенциал ионизации можно оценить как отрицательную энергию ВЗМО (HOMO):

$$I \approx -\varepsilon_{\text{HOMO}}$$

Аналогично сродство к электрону оценивают по энергии НСМО (LUMO):

$$A \approx -\varepsilon_{\text{LUMO}}$$

Более прямой подход заключается в независимом расчёте энергий нейтральной молекулы, её катиона и аниона с последующим вычислением их разности:

$$I = E(\mathrm{M}^+) - E(\mathrm{M})$$
$$A = E(\mathrm{M}) - E(\mathrm{M}^-)$$

In [ ]:
# O2, триплет (мультиплетность 3) — основное состояние
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "o2_triplet")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! HF ma-def2-TZVP PrintMOs PrintBasis
%pal nprocs 4 end

* xyz 0 3
O   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.207000
*
"""
with open("o2_triplet.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca o2_triplet.inp > o2_triplet.out --oversubscribe

In [ ]:
# O2+, катион (заряд +1, дублет)
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "o2_cation")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! HF ma-def2-TZVP PrintMOs PrintBasis
%pal nprocs 4 end

* xyz 1 2
O   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.207000
*
"""
with open("o2_cation.inp", "w") as f:
    f.write(input_file)
!/opt/orca/orca o2_cation.inp > o2_cation.out --oversubscribe

In [ ]:
# O2-, анион (заряд -1, дублет)
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "o2_anion")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! HF ma-def2-TZVP PrintMOs PrintBasis
%pal nprocs 4 end

* xyz -1 2
O   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.207000
*
"""
with open("o2_anion.inp", "w") as f:
    f.write(input_file)
!/opt/orca/orca o2_anion.inp > o2_anion.out --oversubscribe

In [ ]:
hartree_2_ev = 27.2114

systems = {
    # система        E_HOMO,  E_LUMO,  E_total (FINAL SINGLE POINT ENERGY)
    "O2 триплет":   (None,    None,    None),
    "O2+ катион":   (None,    None,    None),
    "O2- анион":    (None,    None,    None),
}

print("Теорема Купманса (орбитальные энергии O2):")
e_homo, e_lumo, _ = systems["O2 триплет"]
if e_homo is not None and e_lumo is not None:
    print(f"  I ≈ -E_HOMO = {-e_homo * hartree_2_ev:6.2f} эВ")
    print(f"  A ≈ -E_LUMO = {-e_lumo * hartree_2_ev:6.2f} эВ")

print("ΔSCF (разности полных энергий):")
E_n, E_cat, E_an = (systems[k][2] for k in ("O2 триплет", "O2+ катион", "O2- анион"))
if None not in (E_n, E_cat, E_an):
    print(f"  I = E(O2+) - E(O2) = {(E_cat - E_n) * hartree_2_ev:6.2f} эВ")
    print(f"  A = E(O2) - E(O2-) = {(E_n - E_an) * hartree_2_ev:6.2f} эВ")

print("Эксперимент:")
print("  I = 12.0697 ± 0.0002 эВ  https://doi.org/10.1016/0009-2614(89)85005-5")
print("  A = 0.448 ± 0.006 эВ     https://doi.org/10.1021/jp0357323")

# ⚙️ Глава 2. Оптимизация геометрии

В приближении Борна-Оппенгеймера электронная энергия зависит от координат ядер как от параметров. Зависимость $E(\mathbf{R})$ называется **поверхностью потенциальной энергии (ППЭ)**. У одной молекулы из $N$ атомов она зависит от $3N - 6$ внутренних координат ($3N - 5$ для линейных молекул). Для оптимизации геометрии системы в ORCA используется ключ `OPT`. В ходе оптимизации программа изменяет координаты атомов, стремясь найти минимум энергии на поверхности потенциальной энергии.

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "water_opt")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP Opt
%pal nprocs 4 end

* xyz 0 1
O   0.000000   0.000000   0.000000
H   0.000000   0.777817  -0.777817
H   0.000000  -0.777817  -0.777817
*
"""

with open("water_opt.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca water_opt.inp > water_opt.out --oversubscribe

In [ ]:
show("water_opt.out")

## 2.1 Матрица Гесса и частоты колебаний

In [ ]:
# Поверхность потенциальной энергии 
def U(x, y):
    return (+ 2.5 * np.exp(-(x**2 + y**2) / 1.5)                  # центральный холм
            - 2.0 * np.exp(-(x**2 + (y - 3)**2) / 1.2)            # глубокая яма
            - 1.4 * np.exp(-((x + 2.6)**2 + (y + 1.5)**2) / 1.2)  # мелкая яма 1
            - 0.9 * np.exp(-((x - 2.6)**2 + (y + 1.5)**2) / 1.2)  # мелкая яма 2 
            + 0.03 * (x**2 + y**2))                               

grid = np.linspace(-5, 5, 200)
X, Y = np.meshgrid(grid, grid)

fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=U(X, Y),
                         colorscale='RdYlBu_r',
                         opacity=0.9, showscale=False))

fig.update_layout(width=850, height=650,
                  scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='U(x, y)'),
                  scene_camera=dict(eye=dict(x=1.3, y=-1.5, z=1.1)))
fig.show()

Одноточечный расчёт позволяет определить энергию $E$ для заданной геометрии молекулы, то есть получить одну точку на поверхности потенциальной энергии. Однако произвольная геометрия, как правило, не соответствует стационарной точке. В стационарной точке градиент энергии равняется нулю:

$$ \nabla E = \left( \frac{\partial E}{\partial R_1},\ \frac{\partial E}{\partial R_2},\ \ldots \right) = 0 $$

Тип стационарной точки определяют по **матрице Гесса** (матрице вторых производных энергии по координатам атомов):

$$ H_{ij} = \frac{\partial^{2} E}{\partial R_i\, \partial R_j} $$

Диагонализация массово-взвешенной матрицы Гесса позволяет получить частоты нормальных колебаний.

| Частоты колебаний | Что это | Интерпретация |
|---|---|---|
| все вещественные | локальный минимум | устойчивая конфигурация |
| ровно одна мнимая частота | седловая точка 1-го порядка | переходное состояние реакции |
| две и более мнимых частоты | седловая точка высшего порядка | физического смысла не имеет |

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "water_opt_freq")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP Opt Freq
%pal nprocs 4 end

* xyz 0 1
O   0.000000   0.000000   0.000000
H   0.000000   0.777817  -0.777817
H   0.000000  -0.777817  -0.777817
*
"""

with open("water_opt_freq.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca water_opt_freq.inp > water_opt_freq.out --oversubscribe

In [ ]:
show("water_opt_freq.out")

### 2.1.1 Гармоническое приближение

При расчёте колебательных частот молекулы потенциальную энергию вблизи равновесной геометрии раскладывают в ряд Тейлора по смещениям ядер и ограничиваются квадратичными членами. Это соответствует модели гармонического осциллятора, в которой сила линейно зависит от смещения. Колебательные частоты в таком приближении определяются массами атомов и кривизной поверхности потенциальной энергии в точке равновесия.

Реальный потенциал не квадратичен, поэтому расчётные гармонические частоты систематически **выше** экспериментальных на несколько процентов. На практике рассчитанные частоты колебаний умножают на эмпирический **масштабирующий множитель** (scaling factor), для B3LYP это около 0.96-0.97. Этот множитель позволяет приблизить гармонические частоты к экспериментальным частотам колебаний.

В ORCA масштабирование частот можно задать следующим образом:
```
%freq
  ScalFreq 0.96
end
```
* Harmonic Scale Factors of Fundamental Transitions for Dispersion‐corrected Quantum Chemical Methods: https://doi.org/10.1002/cphc.202400547

### 2.1.2 Численный и аналитический расчёт матрицы Гесса

В большинстве распространённых методов ORCA матрица Гесса может быть вычислена аналитически с помощью ключевого слова `Freq`. Если аналитические вторые производные для выбранного метода не реализованы, используют численное дифференцирование градиентов `NumFreq`. Для этого каждый атом последовательно смещают в положительном и отрицательном направлениях вдоль координат (x,y,z), после чего рассчитывают градиенты энергии. Численный расчёт требует большого количества дополнительных вычислений, и он значительно дороже аналитического. 

## 2.2 Термохимия

Расчёт колебательных частот позволяет не только определить положение системы на поверхности потенциальной энергии, но и определить термодинамические функции текущей конфигурации: энтальпию (H), энтропию (S) и энергию Гиббса (G) при заданных температуре и давлении. В ORCA соответствующие результаты выводятся в блоке `THERMOCHEMISTRY`. По умолчанию молекула рассматривается как идеальный газ при давлении 1 атм в приближениях жёсткого ротатора и гармонических осцилляторов. 

**Энергия нулевых колебаний** (ZPE, Zero-Point Energy). Даже при абсолютном нуле температуры молекула не может находиться в состоянии абсолютного полного покоя и, согласно квантовой механике, колебательная энергия гармонического осциллятора в основном состоянии отлична от нуля. 

$$ E_{\text{ZPE}} = \frac{1}{2}\sum_{k=1}^{N_{кол}} h\,\nu_k $$

**Энтальпия.** К электронной энергии минимума добавляются энергия нулевых колебаний и тепловые поправки:

$$H(T)=E_{\mathrm{elec}}+H_{\mathrm{thermal}}(T)$$
$$
H_{\mathrm{thermal}}(T) = \underbrace{\frac{1}{2}\sum_{k=1}^{N_{\text{кол}}} h\nu_k}_{E_{\mathrm{ZPE}}}
+\underbrace{\sum_{k=1}^{N_{\text{кол}}}\frac{h\nu_k}{e^{h\nu_k/k_BT}-1}}_{E_{\text{кол}}(T)}
+\underbrace{\tfrac{3}{2}k_BT}_{E_{\text{вр}^{линейная}}}
+\underbrace{\tfrac{3}{2}k_BT}_{E_{\text{пост}}}
+k_BT
$$
**Энтропия** может быть рассчитана как сумма вращательного, колебательного и электронного вкладов:

$$ S = S_{\text{пост}}+S_{\text{вр}}+S_{\text{кол}}+S_{\text{эл}} $$

**Энергия Гиббса.** 

$$G(T) = H(T) - TS$$

In [ ]:
# N2
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "reaction_n2")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP TightOpt Freq
%pal nprocs 4 end

* xyz 0 1
N   0.000000   0.000000   0.000000
N   0.000000   0.000000   1.100000
*
"""
with open("reaction_n2.inp", "w") as f:
    f.write(input_file)
!/opt/orca/orca reaction_n2.inp > reaction_n2.out --oversubscribe

In [ ]:
# H2
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "reaction_h2")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP TightOpt Freq
%pal nprocs 4 end

* xyz 0 1
H   0.000000   0.000000   0.000000
H   0.000000   0.000000   0.740000
*
"""
with open("reaction_h2.inp", "w") as f:
    f.write(input_file)
!/opt/orca/orca reaction_h2.inp > reaction_h2.out --oversubscribe

In [ ]:
# NH3
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "reaction_nh3")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP TightOpt Freq
%pal nprocs 4 end

* xyz 0 1
N   0.000000   0.000000   0.000000
H   0.937500   0.000000  -0.381000
H  -0.468750   0.811900  -0.381000
H  -0.468750  -0.811900  -0.381000
*
"""
with open("reaction_nh3.inp", "w") as f:
    f.write(input_file)
!/opt/orca/orca reaction_nh3.inp > reaction_nh3.out --oversubscribe

In [ ]:
# N2 + 3H2 = 2NH3
# Значения из блока THERMOCHEMISTRY при 298.15 K, в хартри:
#   E(ele) - Electronic energy
#   ZPE    - Zero point energy
#   H      - Total Enthalpy
#   S      - Final entropy term  (в ORCA это уже T·S)
#   G      - Final Gibbs free energy

N2 = {
    "E(ele)": None,
    "ZPE":    None,
    "H":      None,
    "S":      None,
    "G":      None,
}

H2 = {
    "E(ele)": None,
    "ZPE":    None,
    "H":      None,
    "S":      None,
    "G":      None,
}

NH3 = {
    "E(ele)": None,
    "ZPE":    None,
    "H":      None,
    "S":      None,
    "G":      None,
}

hartree_2_kj = 2625.4996
T = 298.15
R = 8.314462e-3 # кДж/(моль·K)

def delta(key):
    """ΔX = 2·X(NH3) - X(N2) - 3·X(H2), кДж/моль"""
    return (2 * NH3[key] - N2[key] - 3 * H2[key]) * hartree_2_kj

dE   = delta("E(ele)")
dZPE = delta("ZPE")
dH   = delta("H")
dTS  = delta("S")
dG   = delta("G")
print(f"ΔE(ele) = {dE:8.1f} кДж/моль")
print(f"ΔZPE    = {dZPE:8.1f} кДж/моль")
print(f"ΔH      = {dH:8.1f} кДж/моль")
print(f"TΔS     = {dTS:8.1f} кДж/моль")
print(f"ΔG      = {dG:8.1f} кДж/моль")
print(f"K_p     = {math.exp(-dG / (R * T)):.3g}")

**Экспериментальные данные** (NIST Chemistry WebBook, 298.15 K, 1 бар):

| Молекула | $\Delta_f H^\circ$, кДж/моль | $S^\circ$, Дж/(моль·K) |
|---|---|---|
| N₂ | 0 | 191.609 ± 0.004 |
| H₂ | 0 | 130.680 ± 0.003 |
| NH₃ | −45.94 ± 0.35 | 192.77 ± 0.05 |

$\Delta_r H^\circ = 2\,\Delta_f H^\circ(\mathrm{NH_3}) = -91.88 \text{ кДж/моль}$

$\Delta_r S^\circ = 2\,S^\circ(\mathrm{NH_3}) - S^\circ(\mathrm{N_2}) - 3\,S^\circ(\mathrm{H_2}) = -198.1$ Дж/(моль·K)

$\Delta_r G^\circ (298.15) = \Delta_r H^\circ - 298.15⋅\Delta_r S^\circ = -32.81 \text{ кДж/моль}$

$K_p = e^{-\Delta_r G^\circ / RT} \approx 5.6\cdot10^{5}$


## 2.3 Оптимизация с ограничениями

### 2.3.1 Геометрические ограничения

В ORCA ограничения задаются в блоке `%geom`, каждое ограничение отдельной строкой в фигурных скобках:

```
%geom
  Constraints
    { B 0 1 1.45 C }      # связь между атомами 0-1 зафиксирована на 1.45 Å
    { A 2 0 1 100.0 C }   # валентный угол 2–0–1 на 100°
    { D 2 0 1 3 90.0 C }  # торсион 2–0–1–3 на 90°
    { C 3 C }             # атом 3 заморожен в пространстве (декартовы координаты)
  end
end
```

* нумерация атомов в ORCA начинается **с нуля**.
* если значение не указано (`{ B 0 1 C }`), координата фиксируется на **стартовом** значении.

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "nh3_planar")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP Opt Freq
%pal nprocs 4 end

%geom
  Constraints
    { A 1 0 2 120.0 C }
    { A 2 0 3 120.0 C }
    { A 1 0 3 120.0 C } 
  end
end

* xyz 0 1
N   0.000000   0.000000   0.200000
H   1.010000   0.000000   0.000000
H  -0.505000   0.874689   0.000000
H  -0.505000  -0.874689   0.000000
*
"""

with open("nh3_planar.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca nh3_planar.inp > nh3_planar.out --oversubscribe

In [ ]:
show("nh3_planar.out")

### 2.3.2 Симметрия молекулы в расчёте

По умолчанию симметрия в ORCA выключена. Ключевое слово `UseSym` включает её, и ORCA определяет группу симметрии автоматически. Группу симметрии можно задать и вручную, в блоке `%sym`:

```
%sym
  PointGroup "D2h"
end
```

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "benzene_d6h")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP UseSym Opt
%pal nprocs 4 end

%sym
  PointGroup "D6h"
end

* xyz 0 1
C   1.390000   0.000000   0.000000
C   0.695000   1.203775   0.000000
C  -0.695000   1.203775   0.000000
C  -1.390000   0.000000   0.000000
C  -0.695000  -1.203775   0.000000
C   0.695000  -1.203775   0.000000
H   2.470000   0.000000   0.000000
H   1.235000   2.139083   0.000000
H  -1.235000   2.139083   0.000000
H  -2.470000   0.000000   0.000000
H  -1.235000  -2.139083   0.000000
H   1.235000  -2.139083   0.000000
*
"""

with open("benzene_d6h.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca benzene_d6h.inp > benzene_d6h.out --oversubscribe

In [ ]:
show("benzene_d6h.out")

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "benzene_d2h")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP UseSym PrintMOs PrintBasis
%pal nprocs 4 end

%sym
  PointGroup "D2h"
end

* xyz 0 1
C   1.501600   0.000000   0.000000
C   0.670000   1.200000   0.000000
C  -0.670000   1.200000   0.000000
C  -1.501600   0.000000   0.000000
C  -0.670000  -1.200000   0.000000
C   0.670000  -1.200000   0.000000
H   2.581600   0.000000   0.000000
H   1.171000   2.157000   0.000000
H  -1.171000   2.157000   0.000000
H  -2.581600   0.000000   0.000000
H  -1.171000  -2.157000   0.000000
H   1.171000  -2.157000   0.000000
*
"""

with open("benzene_d2h.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca benzene_d2h.inp > benzene_d2h.out --oversubscribe

In [ ]:
show("benzene_d2h.out")

## 2.4 Полуэмпирические методы 

**Идея.** Сохранить формализм метода Хартри-Фока (матрица Фока, SCF-цикл), но попытаться упростить вычисления, используя дополнительные приближения:

1. Рассматриваются только валентные электроны. Остовные электроны явно не включают в расчёт, а их влияние описывают эффективным потенциалом.
2. Используют минимальный набор базисных функций, соответствующий валентным атомным орбиталям.
3. Большинство двухэлектронных интегралов **обнуляется** (приближение NDDO, Neglect of Diatomic Differential Overlap, пренебрежение двухатомным дифференциальным перекрыванием).

В итоге число базисных функций $M$ мало, число интегралов резко сокращается, оставшиеся параметризуют, и расчёт идёт на порядки быстрее классических методов HF и DFT. 

Классические методы NDDO различаются параметризацией: MNDO, AM1, PM3, PM6 и PM7. В ORCA доступны `MNDO`, `AM1`, `PM3`.

**Методы сильной связи: GFN-xTB**

Реализация от группы Гримме (https://doi.org/10.1021/acs.jctc.7b00118): *Geometries, Frequencies, Non-covalent interactions*. Энергия раскладывается следующим образом:

$$ E = E_{\text{отт}} + E_{\text{EHT}} + E_{\text{эл}} + E_{\text{дисп}} + E_{\text{попр}} $$

* $E_{\text{отт}}$ - отталкивание атомных остовов (ядро плюс внутренние электроны, заданные эффективным потенциалом);
* $E_{\text{EHT}}$ - одноэлектронная энергия в минимальном валентном базисе;
* $E_{\text{эл}}$ - кулоновское взаимодействие частичных атомных зарядов;
* $E_{\text{дисп}}$ - дисперсионная поправка;
* $E_{\text{попр}}$ - эмпирические поправки для водородных и галогенных связей.

Параметры задаются поэлементно, единым набором для всех элементов до радона ($Z \le 86$).

Последняя версия метода GFN2-xTB (https://doi.org/10.1021/acs.jctc.8b01176)

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "caffeine_xtb")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! XTB2 Opt NumFreq
%pal nprocs 4 end

* xyz 0 1
C    3.174832  -1.011051   0.072092
N    2.158411   0.010178   0.074031
C    2.370251   1.362472   0.129762
N    1.235410   2.033366   0.115094
C    0.273691   1.071029   0.047757
C    0.807625  -0.181441   0.021058
C    0.020340  -1.362531  -0.048173
O    0.511676  -2.487056  -0.071735
N   -1.351642  -1.096783  -0.085470
C   -2.263798  -2.221383  -0.156941
C   -1.942544   0.184127  -0.059694
O   -3.169527   0.317256  -0.096796
N   -1.083346   1.281332   0.008986
C   -1.610868   2.633177   0.039469
H    3.087423  -1.587441  -0.852237
H    3.026096  -1.651891   0.944704
H    4.161531  -0.543464   0.122538
H    3.358799   1.801771   0.179255
H   -2.871038  -2.129720  -1.063507
H   -1.745482  -3.183232  -0.173754
H   -2.931580  -2.193352   0.710405
H   -1.286963   3.122408   0.963565
H   -2.703287   2.645747   0.002650
H   -1.226008   3.186482  -0.823057
*
"""

with open("caffeine_xtb.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca caffeine_xtb.inp > caffeine_xtb.out --oversubscribe

In [ ]:
show("caffeine_xtb.out")

# 🕵️‍♂️ Глава 3. Сканирование поверхности потенциальной энергии

Оптимизация геометрии - это способ найти ближайший минимум на поверхности потенциальной энергии. Но исследователю часто бывает важна еще и **форма ППЭ** вдоль какой-то координаты: как меняется энергия, например, при вращении вокруг связи, при растяжении связи до разрыва, при сближении двух молекул. Для этого ППЭ **сканируют**: выбранную внутреннюю координату $q$ меняют с заданным шагом и в каждой точке считают энергию. Результат - **профиль энергии** $E(q)$.

Два способа получить профиль:
* **жёсткий скан** (rigid, unrelaxed): меняется только $q$, все остальные координаты заморожены. Профиль энергии в нерелаксированном сканировании лежит не ниже релаксированного: $E_\text{rigid}(q) \ge E_\text{relaxed}(q)$.
* **релаксированный скан** (relaxed): в каждой точке $q$ фиксируется, а по всем остальным координатам геометрия **оптимизируется**.

In [ ]:
# Гомолитический разрыв связи в молекуле H2 (RHF)
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "h2_rhf_scan")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! RHF def2-SVP OPT
%pal nprocs 4 end

%geom
  Scan B 0 1 = 0.5, 4.0, 15 end
end

* xyz 0 1
H   0.000000   0.000000   0.000000
H   0.000000   0.000000   0.740000
*
"""

with open("h2_rhf_scan.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca h2_rhf_scan.inp > h2_rhf_scan.out --oversubscribe

In [ ]:
show("h2_rhf_scan.out")

In [ ]:
# Вращение вокруг O-O в H2O2 (пероксид водорода)
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "h2o2_scan")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP Opt
%pal nprocs 4 end

%geom
  Scan D 2 0 1 3 = 0, 180, 6 end
end

* xyz 0 1
O   0.000000   0.000000   0.000000
O   0.000000   0.000000   1.450000
H   0.955300   0.000000  -0.168400
H   0.955300   0.000000   1.618400
*
"""

with open("h2o2_scan.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca h2o2_scan.inp > h2o2_scan.out --oversubscribe

In [ ]:
show("h2o2_scan.out")

# ♻️ Глава 4. Поиск переходного состояния 

Для поиска переходного состояния в ORCA используется ключевое слово `OptTS`. В отличие от обычной оптимизации, которая ищет минимум энергии, этот алгоритм ищет седловую точку первого порядка на поверхности потенциальной энергии (ППЭ). 

В данном случае оптимизатор делает шаги вдоль одного направления - собственного вектора гессиана с **наименьшим** (отрицательным) собственным значением, при котором энергия максимизируется, а вдоль всех остальных направлений - минимизируется. Такой алгоритм называется **следованием за собственным вектором** (eigenvector following).

Для успешной работы OptTS начальная геометрия должна находиться достаточно близко к искомому переходному состоянию. Ключ `Calc_Hess true` в блоке `%geom` считает гессиан в стартовой точке, чтобы определить направление поиска. 

Поскольку матрица Гесса меняется по мере движения по ППЭ, его периодически пересчитывают. Например, `Recalc_Hess 5` задаёт пересчёт гессиана каждые пять шагов оптимизации. 

**Проверка переходного состояния.** После оптимизации необходимо выполнить расчёт колебательных частот с помощью `Freq`. У геометрии переходного состояния должна быть **ровно одна** мнимая частота, и при этом соответствующее нормальное колебание должно описывать предполагаемое превращение.

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "hcn_ts")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP OptTS Freq
%pal nprocs 4 end

%geom
  Calc_Hess true
end

* xyz 0 1
C   0.000000   0.000000   0.000000
N   1.190000   0.000000   0.000000
H   0.450000   1.150000   0.000000
*
"""

with open("hcn_ts.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca hcn_ts.inp > hcn_ts.out --oversubscribe

In [ ]:
show("hcn_ts.out")

## 4.1 Метод упругой ленты (NEB)
Если геометрия переходного состояния неизвестна и трудно построить подходящее начальное приближение для OptTS, можно использовать метод упругой ленты (NEB, nudged elastic band). NEB не требует геометрии, близкой к переходному состоянию, как при OptTS, но требует геометрии начальной (реагенты) и конечной (продукты) конфигурации системы. 

Метод строит цепочку промежуточных структур, называемых изображениями (images), между начальным и конечным состояниями. Каждое изображение представляет собой отдельную (промежуточную) конфигурацию системы. В ходе оптимизации изображения перемещаются под действием сил, обусловленных формой поверхности потенциальной энергии, и фиктивных упругих связей между соседними изображениями. 

В результате получается дискретное представление пути реакции, по которому можно оценить энергетический барьер и определить область, соответствующую переходному состоянию. Найденное высокоэнергетическое изображение, близкое к переходному состоянию, желательно дополнительно оптимизировать и проверить расчётом колебательных частот. 

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "mal_react")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP Opt
%pal nprocs 4 end

* xyz 0 1
O   -1.177795  -0.680000   0.000000
C   -1.177795   0.680000   0.000000
C    0.000000   1.360000   0.000000
C    1.177795   0.680000   0.000000
O    1.177795  -0.680000   0.000000
H   -0.329090  -1.170000   0.000000
H   -2.113102   1.220000   0.000000
H    0.000000   2.440000   0.000000
H    2.113102   1.220000   0.000000
*
"""

with open("mal_react.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca mal_react.inp > mal_react.out --oversubscribe

In [ ]:
show("mal_react.out")

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "mal_prod")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP Opt
%pal nprocs 4 end

* xyz 0 1
O   -1.177795  -0.680000   0.000000
C   -1.177795   0.680000   0.000000
C    0.000000   1.360000   0.000000
C    1.177795   0.680000   0.000000
O    1.177795  -0.680000   0.000000
H    0.329090  -1.170000   0.000000
H   -2.113102   1.220000   0.000000
H    0.000000   2.440000   0.000000
H    2.113102   1.220000   0.000000
*
"""

with open("mal_prod.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca mal_prod.inp > mal_prod.out --oversubscribe

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "mal_neb")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-SVP NEB-TS Freq
%pal nprocs 4 end

%neb
  NEB_End_XYZFile "../mal_prod/mal_prod.xyz"
  NImages 8
end

* xyzfile 0 1 ../mal_react/mal_react.xyz
"""

with open("mal_neb.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca mal_neb.inp > mal_neb.out --oversubscribe

In [ ]:
show("mal_neb.out", "mal_neb_MEP_trj.xyz")

# 🤖 Глава 5. Дизайн и анализ вычислительных экспериментов

## 5.1 Композитные методы 

Точность квантово-химического расчёта определяется не только выбранным методом, но и базисным набором, качеством оптимизированной геометрии и учётом термодинамических поправок. При этом наиболее точные методы часто оказываются слишком дорогими, чтобы использовать их на всех этапах расчёта. Один из способов снизить вычислительные затраты - разделить расчёт на несколько этапов, используя для каждого из них подходящий метод.

### 5.1.1 Разделение геометрии и электронной энергии

Важный принцип состоит в том, что геометрия молекулы и её электронная энергия могут рассчитываться разными методами. Оптимизация геометрии требует многократного вычисления полной энергии и её градиента, тогда как для уточнения энергии (электронной корреляции) достаточно выполнить отдельный расчёт на уже найденной геометрии.

Например, запись `CCSD(T)/aug-cc-pVQZ//B3LYP-D3(BJ)/def2-TZVP` указывает на то, что геометрия была оптимизирована с использованием метода B3LYP с базисным набором def2-TZVP и дисперсионной поправкой D3BJ, а после одноточечная энергия была рассчитана с использованием этой геометрии более дорогим методом связанных кластеров CCSD(T) с базисным набором aug-cc-pVQZ. 

В принятой в квантовой химии записи метод слева от двойной косой черты соответствует расчёту энергии, а справа - методу оптимизации геометрии.

### 5.1.2 Энергия Гиббса 

Энергию Гиббса, рассчитываемую методами квантовой химии, можно представить в виде уравнения:
$$ G(T) = E_{\mathrm{elec}}+ H_{\mathrm{thermal}}(T) - TS$$

В таком случае можно рассчитать термодинамические поправки ($H_{\mathrm{thermal}}(T)$ и $S$) после оптимизации геометрии более дешёвым методом, а электронную энергию $E_{\mathrm{elec}}$ более дорогим методом. Этот подход позволяет повысить точность оценки электронной энергии без необходимости выполнять дорогостоящую оптимизацию геометрии и расчёт колебательных частот на том же уровне теории.

HEAT (High accuracy Extrapolated Ab initio Thermochemistry) protocol: https://doi.org/10.1063/1.1811608
$$E_{HEAT} = E_{HF}^{\infty} + {\Delta}E_{CCSD(T)}^{\infty} +{\Delta}E_{CCSDT} + {\Delta}E_{CCSDTQ} + {\Delta}E_{rel} + {\Delta}E_{ZPE} + {\Delta}E_{DBOC} + {\Delta}E_{SOC}$$

## 5.2 QTAIM

Теория Бейдера «атомы в молекулах» (QTAIM) позволяет исследовать электронную плотность $\rho(\mathbf r)$ и на её основе описывать атомы и химические связи в молекулах. Критические точки электронной плотности - это точки, в которых её градиент равен нулю: $\nabla\rho(\mathbf r)=0$. Их классифицируют по сигнатуре $(3,\omega)$, где первое значение соответствует трём ненулевым собственным значениям гессиана, а второе - числу его отрицательных собственных значений минус число положительных.

| Тип | Что это |
|---|---|
| (3, −3) | максимум - ядро |
| (3, −1) | седло между двумя ядрами (BCP, Bond Critical Point)|
| (3, +1) | критическая точка кольца |
| (3, +3) | критическая точка клетки |

В QTAIM наличие BCP между атомами рассматривается как признак их топологической связи в распределении электронной плотности, но не является универсальным доказательством существования химической связи. Тем не менее, можно попробовать описать характер такого взаимодействия по лапласиану $\nabla^2\rho$ в критической точке связи: 

| | $\nabla^2\rho$ |
|---|---|
| ковалентная | $< 0$ |
| ионная, водородная, ван-дер-ваальсова| $> 0$ |

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "qtaim_c6h6")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP PrintMOs PrintBasis
%pal nprocs 4 end

* xyz 0 1
C   1.390000   0.000000   0.000000
C   0.695000   1.203775   0.000000
C  -0.695000   1.203775   0.000000
C  -1.390000   0.000000   0.000000
C  -0.695000  -1.203775   0.000000
C   0.695000  -1.203775   0.000000
H   2.470000   0.000000   0.000000
H   1.235000   2.139083   0.000000
H  -1.235000   2.139083   0.000000
H  -2.470000   0.000000   0.000000
H  -1.235000  -2.139083   0.000000
H   1.235000  -2.139083   0.000000
*
"""
with open("qtaim_c6h6.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca qtaim_c6h6.inp > qtaim_c6h6.out --oversubscribe

# волновая функция: qtaim_c6h6.gbw -> qtaim_c6h6.wfn (и .wfx)
ORCA_2AIM = os.path.join(os.path.dirname(ORCA), "orca_2aim")
!{ORCA_2AIM} qtaim_c6h6 > qtaim_c6h6.2aim.log

In [ ]:
show("qtaim_c6h6.out", "qtaim_c6h6.wfn", qtaim=True)

## 5.3 Функция электронной локализации (ELF)

Функция электронной локализации (ELF, Electron Localization Function) позволяет анализировать распределение электронов и выявлять области их пространственной локализации. В отличие от QTAIM, основанной на топологии электронной плотности $\rho(\mathbf r)$, ELF характеризует локализацию электронов с учётом их кинетической энергии.

Значения ELF лежат в диапазоне от 0 до 1:
* $ELF \approx 1$ - сильная локализация электронов, характерная, например, для неподелённых электронных пар и ковалентных связей.
* $ELF \approx 0.5$ - локализация, характерная для однородного электронного газа.
* $ELF \approx 0$ - области слабой электронной локализации.

Анализ ELF позволяет выделять пространственные области, соответствующие электронным парам, и исследовать характер химических связей. Например, между двумя атомами может формироваться область высокой ELF, соответствующая локализованной электронной паре ковалентной связи, а около атома - области, связанные с неподелёнными электронными парами.

In [ ]:
main_path = "/home/jupyter/project/computational-chemistry-course/Занятия/Занятие_5"
work_path = os.path.join(main_path, "elf_water")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

input_file = """\
! B3LYP def2-TZVP PrintMOs PrintBasis
%pal nprocs 4 end

* xyz 0 1
O   0.000000   0.000000   0.117300
H   0.000000   0.757200  -0.469200
H   0.000000  -0.757200  -0.469200
*
"""
with open("elf_water.inp", "w") as f:
    f.write(input_file)

!/opt/orca/orca elf_water.inp > elf_water.out --oversubscribe

# волновая функция: elf_water.gbw -> elf_water.wfn (и .wfx)
ORCA_2AIM = os.path.join(os.path.dirname(ORCA), "orca_2aim")
!{ORCA_2AIM} elf_water > elf_water.2aim.log

In [ ]:
show("elf_water.out", "elf_water.wfn", elf=True)

# 📚 Дополнительные материалы

**Визуализация молекулярных структур и результатов расчётов**:
* *Avogadro* - построение и редактирование молекулярных структур, подготовка входных файлов для QM расчётов и визуализация молекулярных свойств.
* *Chemcraft* - визуализация молекулярных орбиталей, колебательных мод, электронных плотностей и других результатов QM расчётов.
* *ChimeraX + SEQCROW* - визуализация молекулярных структур и анализ результатов QM расчётов непосредственно в *ChimeraX* с помощью расширения *SEQCROW*.

**Анализ результатов расчётов**:
* *Multiwfn* - многофункциональный инструмент для анализа электронной плотности, молекулярных орбиталей, ELF, QTAIM и других электронных свойств.
* *orca_plot, orca_2mkl, orca_vib* - утилиты ORCA для визуализации молекулярных свойств на пространственной сетке, преобразования результатов расчётов и анализа колебательных характеристик соответственно.
* *cclib* - библиотека Python для извлечения и обработки структурированных данных из выходных файлов различных квантово-химических программ.
* *NBO (Natural Bond Orbital analysis)* - анализ естественных атомных и связывающих орбиталей, распределения электронной плотности и донорно-акцепторных взаимодействий.